In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/prishabhkumar/email-intelligence-processed-data/Cleaned data/bc3_threads.json
/kaggle/input/datasets/prishabhkumar/email-intelligence-processed-data/Cleaned data/enron_cleaned.csv
/kaggle/input/datasets/prishabhkumar/email-intelligence-processed-data/Cleaned data/spamassassin_cleaned.csv
/kaggle/input/datasets/prishabhkumar/email-intelligence-processed-data/val/spamassassin_val.csv
/kaggle/input/datasets/prishabhkumar/email-intelligence-processed-data/val/enron_val.csv
/kaggle/input/datasets/prishabhkumar/email-intelligence-processed-data/test/spamassassin_test.csv
/kaggle/input/datasets/prishabhkumar/email-intelligence-processed-data/test/enron_test.csv
/kaggle/input/datasets/prishabhkumar/email-intelligence-processed-data/train/spamassassin_train.csv
/kaggle/input/datasets/prishabhkumar/email-intelligence-processed-data/train/enron_train.csv
/kaggle/input/datasets/marawanxmamdouh/email-thread-summary-dataset/CSV/email_thread_details.csv
/kaggle/input/datasets/m

# Email thread summariser: train `t5-small` and export for Android

Run the cells in order on Kaggle (GPU on, internet on). Everything here is for the **summariser** only. The intent and NER models are already trained.

**Pipeline:** load data, clean emails, build one shortened text per thread, split by thread, fine-tune `t5-small`, evaluate with ROUGE, export to ONNX int8.

Why `t5-small`: it is about 60M parameters, so it is roughly 60-80 MB after int8 quantisation, compared with 496 MB for your quantised T5 Long.

## Cell 1: install extra libraries
`evaluate` and `rouge_score` compute ROUGE. `optimum` handles the ONNX export and quantisation. `transformers`, `datasets` and `torch` are already on Kaggle.

In [2]:
!pip install -q evaluate rouge_score "optimum[onnxruntime]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 911.1 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 4.2 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 68.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 79.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 82.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 7.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingf

## Cell 2: configuration and loading
Sets the file paths, the column names and the length limits used by every later cell.

- `details` has one row per email (`thread_id, subject, timestamp, from, to, body`).
- `summaries` has one summary per thread (`thread_id, summary`).
- `enron` and `spamassassin` are not used here (no summaries). Keep them for testing the full pipeline.

In [3]:
import re, inspect, numpy as np, pandas as pd, torch

BASE = "/kaggle/input/datasets/marawanxmamdouh/email-thread-summary-dataset/CSV/"
ID, SENDER, BODY = "thread_id", "from", "body"
S_ID, S_TEXT = "thread_id", "summary"

TOTAL_WORDS = 550   # was 700
MAX_IN  = 1024
MAX_OUT = 224       # was 192

details = pd.read_csv(BASE + "email_thread_details.csv")
summ    = pd.read_csv(BASE + "email_thread_summaries.csv")
print(details.shape, details[ID].nunique(), "threads in details")
print(summ.shape, "summaries")

(21684, 6) 4167 threads in details
(4167, 2) summaries


## Cell 3: clean emails and build one text per thread
1. `clean()` removes quoted replies, "Original Message" blocks, "On ... wrote:" blocks and signatures, then collapses whitespace.
2. Emails are sorted by time inside each thread and written as `name: message`.
3. `build_thread()` gives every message an equal word budget (`TOTAL_WORDS // number_of_messages`, minimum 25 words). This keeps the start of every message, where the request or decision usually is, instead of cutting off the end of the thread.
4. The subject is added at the front and the T5 prefix `summarize:` is added.

**Use the same `clean()` and `build_thread()` logic on the phone**, otherwise training and inference inputs will differ.

In [4]:
def clean(t):
    t = re.sub(r"=\s*\n", "", t)
    t = str(t)
    t = re.sub(r"(?s)-----Original Message-----.*", "", t)
    t = re.sub(r"(?s)On .{0,80}wrote:.*", "", t)
    t = re.sub(r"(?m)^>.*$", "", t)
    t = re.sub(r"(?s)(--\s*\n|Sent from my).*", "", t)
    return re.sub(r"\s+", " ", t).strip()

details[BODY] = details[BODY].fillna("").map(clean)
details = details[details[BODY].str.len() > 0].copy()
details["ts"] = pd.to_datetime(details["timestamp"], errors="coerce", utc=True)
details = details.sort_values([ID, "ts"], kind="stable")
details["name"] = details[SENDER].astype(str).str.split("@").str[0]
details["line"] = details["name"] + ": " + details[BODY]

MAX_MSGS = 12  # keep the start and end of very long threads

def build_thread(lines):
    if len(lines) > MAX_MSGS:
        half = MAX_MSGS // 2
        lines = lines[:half] + lines[-half:]
    per = max(10, TOTAL_WORDS // len(lines))
    out = []
    for m in lines:
        w = m.split()
        out.append(" ".join(w[:per]) + (" ..." if len(w) > per else ""))
    return " ".join(out)

rows = []
for tid, g in details.groupby(ID, sort=False):
    rows.append({ID: tid, "subject": g["subject"].iloc[0], "text": build_thread(g["line"].tolist())})
threads = pd.DataFrame(rows)

summ = summ.drop_duplicates(S_ID)
data = threads.merge(summ[[S_ID, S_TEXT]], on=ID)
data["text"] = "summarize: " + data["subject"].fillna("").astype(str) + " | " + data["text"]
print(data.shape)
print(data.iloc[0]["text"][:600])
print("SUMMARY:", data.iloc[0][S_TEXT][:300])

(4140, 4)
summarize: FW: Master Termination Log | Theriot, Kim S. KTHERIO: Note to Stephanie Panus.... Stephanie...please remove my name as well as Melissa Murphy's from the distribution list below. Please add the following: Todd D. Hall Kevin Sweeney Rita Wynne Rebecca Grace Rhonda Robinson Kerri Thomspon Kristin Albrecht Tom Chapman Thanks! Kim Theriot
SUMMARY: The email thread discusses the Master Termination Log and the need to investigate a CNG LDC (Hope Gas) termination and a $66 million settlement offer. Stephanie Panus sends out the Daily List and Master Termination Log for various dates. Kim Theriot requests her name and Melissa Murphy's name to be 


## Cell 4: check token lengths
Confirms the shortened threads fit in `MAX_IN` tokens. **You want input p95 at or below 1024.** If it is higher, lower `TOTAL_WORDS` to 600 in Cell 2 and rerun Cell 3.

In [5]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("t5-small")

s = data.sample(min(500, len(data)), random_state=0)
x = [len(tok(t).input_ids) for t in s["text"]]
y = [len(tok(t).input_ids) for t in s[S_TEXT]]
print("input  p50/90/95/99:", np.percentile(x, [50, 90, 95, 99]))
print("output p50/90/95/99:", np.percentile(y, [50, 90, 95, 99]))
if np.percentile(x, 95) > MAX_IN:
    print("WARNING: p95 input is above MAX_IN. Lower TOTAL_WORDS and rerun Cell 3.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (934 > 512). Running this sequence through the model will result in indexing errors


input  p50/90/95/99: [ 407.5   887.    995.15 1162.47]
output p50/90/95/99: [114.   153.3  168.05 224.02]


## Cell 5: split by thread and tokenise
Each row is one thread, so splitting rows gives a split by thread (no thread in two sets). Split is 80% train, 10% validation, 10% test. The test set is saved to `test_split.csv` for later checks.

In [6]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

tr, tmp = train_test_split(data, test_size=0.2, random_state=42)
va, te  = train_test_split(tmp, test_size=0.5, random_state=42)
te = te.reset_index(drop=True)
te.to_csv("test_split.csv", index=False)
print(len(tr), len(va), len(te))

def enc(b):
    x = tok(b["text"], max_length=MAX_IN, truncation=True)
    x["labels"] = tok(text_target=b[S_TEXT], max_length=MAX_OUT, truncation=True)["input_ids"]
    return x

def to_ds(d):
    d = d[["text", S_TEXT]].reset_index(drop=True)
    return Dataset.from_pandas(d, preserve_index=False).map(enc, batched=True, remove_columns=["text", S_TEXT])

ds = {"train": to_ds(tr), "val": to_ds(va), "test": to_ds(te)}

3312 414 414


Map:   0%|          | 0/3312 [00:00<?, ? examples/s]

Map:   0%|          | 0/414 [00:00<?, ? examples/s]

Map:   0%|          | 0/414 [00:00<?, ? examples/s]

## Cell 6: fine-tune `t5-small`
- Batch size 4 with gradient accumulation 4 gives an effective batch of 16 at 1024 tokens. If you get an out-of-memory error, use batch size 2 and accumulation 8.
- `fp16` is off because T5 can produce NaN losses in fp16.
- Validation uses loss only (faster). The best epoch by validation loss is kept. ROUGE comes in Cell 7.
- Learning rate 3e-4 is a normal value for `t5-small`.

In [7]:
import torch
print(torch.cuda.is_available(), torch.cuda.device_count())

True 2


In [8]:
from transformers import (AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments)

model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")
model.config.use_cache = False

key = "eval_strategy" if "eval_strategy" in inspect.signature(Seq2SeqTrainingArguments.__init__).parameters else "evaluation_strategy"
args = Seq2SeqTrainingArguments(
    output_dir="t5-email", num_train_epochs=6,
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    per_device_eval_batch_size=8, learning_rate=3e-4, weight_decay=0.01,
    warmup_ratio=0.05, save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="eval_loss", save_total_limit=2,
    gradient_checkpointing=True, fp16=False, report_to="none",
    **{key: "epoch"})

trainer = Seq2SeqTrainer(model=model, args=args,
    train_dataset=ds["train"], eval_dataset=ds["val"],
    data_collator=DataCollatorForSeq2Seq(tok, model=model))
trainer.train()

model.config.use_cache = True
trainer.save_model("t5-email-small")
tok.save_pretrained("t5-email-small")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,No log,2.394759
2,No log,2.314451
3,No log,2.274887
4,No log,2.250000
5,2.535400,2.242187
6,2.535400,2.238927


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

('t5-email-small/tokenizer_config.json',
 't5-email-small/special_tokens_map.json',
 't5-email-small/spiece.model',
 't5-email-small/added_tokens.json',
 't5-email-small/tokenizer.json')

# SAVE MODEL

In [9]:
import os, shutil
from IPython.display import FileLink

# 1. Zip the fine-tuned PyTorch model (needed if you want to retrain, evaluate or re-export later)
shutil.make_archive("/kaggle/working/t5_email_small_pytorch", "zip", "t5-email-small")

# 3. Delete the training checkpoints so the output stays small
shutil.rmtree("t5-email", ignore_errors=True)

# 4. Show sizes and a download link
for f in os.listdir("/kaggle/working"):
    p = os.path.join("/kaggle/working", f)
    if f.endswith(".zip"):
        print(f"{f}: {os.path.getsize(p) / 1e6:.1f} MB")
display(FileLink("t5_email_small_pytorch.zip"))

t5_email_small_pytorch.zip: 224.1 MB


/kaggle/working/t5_email_small_pytorch.zip

## Cell 7: evaluate with ROUGE on the test set
Generates summaries with beam search and compares them with the reference summaries. ROUGE-1 around 40-50 and ROUGE-L around 30-40 would be a good result. The last lines print three examples so you can read them yourself, which matters as much as the score.

In [13]:
import evaluate
rouge = evaluate.load("rouge")
dev = "cuda" if torch.cuda.is_available() else "cpu"
m = AutoModelForSeq2SeqLM.from_pretrained("t5-email-small").to(dev).eval()

texts, preds = te["text"].tolist(), []
for i in range(0, len(texts), 8):
    b = tok(texts[i:i+8], return_tensors="pt", padding=True,
            truncation=True, max_length=MAX_IN).to(dev)
    with torch.no_grad():
        # out = m.generate(**b, max_new_tokens=MAX_OUT, num_beams=4, no_repeat_ngram_size=3)
        out = m.generate(**b, max_new_tokens=MAX_OUT, num_beams=4,
                 no_repeat_ngram_size=6, length_penalty=1.5, min_new_tokens=60)
    preds += tok.batch_decode(out, skip_special_tokens=True)

scores = rouge.compute(predictions=preds, references=te[S_TEXT].tolist(), use_stemmer=True)
print({k: round(v * 100, 2) for k, v in scores.items()})

for i in range(3):
    print("\nPRED:", preds[i]); print("REF :", te[S_TEXT].iloc[i])

{'rouge1': np.float64(43.3), 'rouge2': np.float64(18.22), 'rougeL': np.float64(27.78), 'rougeLsum': np.float64(27.84)}

PRED: The email thread discusses the renovation of the OU Stadium. The first email mentions that the sooners are above 500. The second email mentions that the earliers are above 500 and that the sooners are below 500. The third email mentions that the OU Stadium renovation was 74,378 as all OSU fans joined in the shitting of bricks. The fourth email mentions the renovation of the stadium and mentions that it is good. The third email is from Timothy Crook, who mentions that it was 74, 378 as all the OSU fans joined in.
REF : Oklahoma University Athletic Director Joe Castiglione announced that the renovations at Owen Field are ahead of schedule and under budget. The materials for the renovation were provided by over 71,000 fans during the fourth quarter of the bedlam football game. Some individuals in the email thread express excitement and support for the renovation, w

# Test decoding settings on a small subset

In [11]:
sub = te.sample(100, random_state=0)
texts, refs = sub["text"].tolist(), sub[S_TEXT].tolist()

def run(**gen):
    preds = []
    for i in range(0, len(texts), 8):
        b = tok(texts[i:i+8], return_tensors="pt", padding=True,
                truncation=True, max_length=MAX_IN).to(dev)
        with torch.no_grad():
            out = m.generate(**b, max_new_tokens=MAX_OUT, num_beams=4, **gen)
        preds += tok.batch_decode(out, skip_special_tokens=True)
    s = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
    return {k: round(v * 100, 2) for k, v in s.items()}, round(np.mean([len(p.split()) for p in preds]))

for name, gen in {
    "ngram3":     dict(no_repeat_ngram_size=3),
    "ngram6":     dict(no_repeat_ngram_size=6),
    "ngram6+len": dict(no_repeat_ngram_size=6, length_penalty=1.5, min_new_tokens=60),
}.items():
    print(name, run(**gen))

ngram3 ({'rouge1': np.float64(43.24), 'rouge2': np.float64(18.03), 'rougeL': np.float64(27.53), 'rougeLsum': np.float64(27.58)}, 62)
ngram6 ({'rouge1': np.float64(41.78), 'rouge2': np.float64(17.59), 'rougeL': np.float64(27.17), 'rougeLsum': np.float64(27.22)}, 63)
ngram6+len ({'rouge1': np.float64(43.35), 'rouge2': np.float64(18.27), 'rougeL': np.float64(27.72), 'rougeLsum': np.float64(27.78)}, 75)


**Re-run Cell 7**

# TESTING

In [15]:
def summarize(subject, messages):
    """messages: list of (sender, body) tuples, oldest first."""
    lines = []
    for sender, body in messages:
        body = clean(body)
        if body:
            lines.append(sender.split("@")[0] + ": " + body)
    text = "summarize: " + subject + " | " + build_thread(lines)
    b = tok(text, return_tensors="pt", truncation=True, max_length=MAX_IN).to(dev)
    with torch.no_grad():
        out = m.generate(**b, max_new_tokens=MAX_OUT, num_beams=4,
                         no_repeat_ngram_size=6, length_penalty=1.5, min_new_tokens=60)
    return tok.decode(out[0], skip_special_tokens=True)

# ---- 1. Your own examples ----
thread = [
    ("alice@company.com", "Hi team, the Q3 budget review is moved to Thursday at 2pm. Please send your numbers by Wednesday noon."),
    ("bob@company.com", "Thursday works for me. I'll send the marketing numbers tomorrow, but the vendor invoice is still pending."),
    ("carol@company.com", "I can't make Thursday 2pm, can we do 3pm? Also, who is presenting the summary to finance?"),
    ("alice@company.com", "3pm is fine. Bob, can you present? Carol, please share the vendor contract by Wednesday."),
]
print(summarize("Q3 budget review", thread))

# ---- 2. Random test-set threads: prediction vs reference ----
for _, r in te.sample(3).iterrows():
    b = tok(r["text"], return_tensors="pt", truncation=True, max_length=MAX_IN).to(dev)
    with torch.no_grad():
        out = m.generate(**b, max_new_tokens=MAX_OUT, num_beams=4,
                         no_repeat_ngram_size=6, length_penalty=1.5, min_new_tokens=60)
    print("\nPRED:", tok.decode(out[0], skip_special_tokens=True))
    print("REF :", r[S_TEXT])

The Q3 budget review is moved to Thursday at 2pm, but the vendor invoice is still pending. The email thread discusses the move to Thursday at 2pm and asks for the numbers to be sent by Wednesday. The email also mentions the need for 3pm and who is presenting the summary to finance. The thread also mentions the need to share the vendor contract by Wednesday.

PRED: The email thread discusses the acquisition of Enron/Delta Turbines. Sheila handled FV and doesn't remember executing any attorney's opinions. Kay Mann asks if there is anything different from the forms she and Danetta worked on earlier. Kent asks if there was someone else who worked on it for the Fountain Valley project. Kay confirms that she has never seen the docs before and mentions that she doesn't have anything on CA Development.
REF : Enron is working on the Delta Power deal, involving the acquisition of four turbines. They are putting the turbines in an LLC and selling some of the interest to Delta Power. Delta Power h

## Cell 8: export to ONNX and quantise to int8
- The first command converts the model to ONNX (encoder, decoder, decoder with cache).
- The second applies int8 dynamic quantisation for ARM64 phones.
- The tokenizer file is copied next to the model because the Android app needs it.

In [16]:
!pip uninstall -y diffusers
!optimum-cli export onnx --model t5-email-small --task text2text-generation-with-past onnx_out/
!optimum-cli onnxruntime quantize --arm64 --onnx_model onnx_out -o onnx_int8
!cp t5-email-small/spiece.model t5-email-small/tokenizer.json t5-email-small/config.json onnx_int8/

Found existing installation: diffusers 0.40.0
Uninstalling diffusers-0.40.0:
  Successfully uninstalled diffusers-0.40.0
`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.13/dist-packages/optimum/exporters/base.py:151: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  self._normalized_config = self.NORMALIZED_CONFIG_CLASS(self._config)
/usr/local/lib/python3.13/dist-packages/transformers/models/t5/modeling_t5.py:1271: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if sequence_length != 1:
/usr/local/lib/python3.13/dist-packages/transformers/cache_utils.py:132: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be 

# test the int8 model against the PyTorch

In [17]:
from optimum.onnxruntime import ORTModelForSeq2SeqLM

mo = ORTModelForSeq2SeqLM.from_pretrained(
    "onnx_int8",
    encoder_file_name="encoder_model_quantized.onnx",
    decoder_file_name="decoder_model_quantized.onnx",
    decoder_with_past_file_name="decoder_with_past_model_quantized.onnx",
)

GEN = dict(max_new_tokens=MAX_OUT, num_beams=4, no_repeat_ngram_size=6,
           length_penalty=1.5, min_new_tokens=60)

for _, r in te.sample(3, random_state=1).iterrows():
    b = tok(r["text"], return_tensors="pt", truncation=True, max_length=MAX_IN)
    with torch.no_grad():
        p = m.generate(**{k: v.to(dev) for k, v in b.items()}, **GEN)
    o = mo.generate(**b, **GEN)
    print("\nPYTORCH:", tok.decode(p[0], skip_special_tokens=True))
    print("INT8   :", tok.decode(o[0], skip_special_tokens=True))


PYTORCH: The email thread discusses the signing of SB 43 by Governor Davis. Scott asks if there is any substance in the "clean-up" provisions that should be aware of. Jeff apologizes for not getting his email address right. He also asks if he can expect the clean-up bill to be signed and if there is substance in the provisions. Scott asks for a view on when the clean-up bill will be signed, and if there is anything of substance in the provisions that need to be aware of.
INT8   : The email thread discusses the signing of SB 43. Jeff asks for a view on when the Clean-Up bill will be signed and if there is substance in the "clean-up" provisions. He also mentions that Governor Davis did sign SBX 43 into law and expects him to sign ABX 43.

PYTORCH: The email thread discusses the role of the Executive Committee as a recurring item in the calendar. The email also mentions the role of a recurring item for the calendar. The thread also includes a discussion about the role of the executive co

# SAVE QUANTASIZED MODEL

In [18]:
import os, shutil
from IPython.display import FileLink

# 2. Zip the int8 ONNX model for Android (only exists after Cell 8)
if os.path.isdir("onnx_int8"):
    shutil.make_archive("/kaggle/working/t5_email_small_onnx_int8", "zip", "onnx_int8")

# 3. Delete the training checkpoints so the output stays small
shutil.rmtree("t5-email", ignore_errors=True)

# 4. Show sizes and a download link
for f in os.listdir("/kaggle/working"):
    p = os.path.join("/kaggle/working", f)
    if f.endswith(".zip"):
        print(f"{f}: {os.path.getsize(p) / 1e6:.1f} MB")
display(FileLink("t5_email_small_onnx_int8.zip"))

t5_email_small_pytorch.zip: 224.1 MB
t5_email_small_onnx_int8.zip: 101.7 MB


/kaggle/working/t5_email_small_onnx_int8.zip

In [20]:
import zipfile, os
keep = ["encoder_model_quantized.onnx", "decoder_model_merged_quantized.onnx",
        "spiece.model", "tokenizer.json", "tokenizer_config.json",
        "special_tokens_map.json", "config.json", "ort_config.json"]
with zipfile.ZipFile("/kaggle/working/t5_email_small_onnx_int8.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in keep:
        z.write(os.path.join("onnx_int8", f), f)
print(os.path.getsize("/kaggle/working/t5_email_small_onnx_int8.zip") / 1e6, "MB")
display(FileLink("t5_email_small_onnx_int8.zip"))

50.096414 MB


/kaggle/working/t5_email_small_onnx_int8.zip

## Cell 9: check file sizes
Compare the int8 folder with your size budget. For `t5-small` expect roughly 60-80 MB for the `.onnx` files together.

In [19]:
!du -sh onnx_out/* onnx_int8/*
!du -sh onnx_int8

4.0K	onnx_out/config.json
160M	onnx_out/decoder_model_merged.onnx
159M	onnx_out/decoder_model.onnx
147M	onnx_out/decoder_with_past_model.onnx
135M	onnx_out/encoder_model.onnx
4.0K	onnx_out/generation_config.json
4.0K	onnx_out/special_tokens_map.json
776K	onnx_out/spiece.model
24K	onnx_out/tokenizer_config.json
2.4M	onnx_out/tokenizer.json
4.0K	onnx_int8/config.json
41M	onnx_int8/decoder_model_merged_quantized.onnx
41M	onnx_int8/decoder_model_quantized.onnx
38M	onnx_int8/decoder_with_past_model_quantized.onnx
34M	onnx_int8/encoder_model_quantized.onnx
4.0K	onnx_int8/ort_config.json
4.0K	onnx_int8/special_tokens_map.json
776K	onnx_int8/spiece.model
24K	onnx_int8/tokenizer_config.json
2.4M	onnx_int8/tokenizer.json
155M	onnx_int8


In [21]:
def summarize_one(subject, sender, body, model=None):
    model = model or m
    line = sender.split("@")[0] + ": " + clean(body)
    text = "summarize: " + subject + " | " + build_thread([line])
    n_in = len(tok(text).input_ids)
    min_new = 20 if n_in < 150 else 40 if n_in < 400 else 60
    b = tok(text, return_tensors="pt", truncation=True, max_length=MAX_IN).to(dev)
    with torch.no_grad():
        out = model.generate(**b, max_new_tokens=MAX_OUT, num_beams=4,
                             no_repeat_ngram_size=6, length_penalty=1.5,
                             min_new_tokens=min_new)
    print(f"[input tokens: {n_in}, min_new_tokens: {min_new}]")
    return tok.decode(out[0], skip_special_tokens=True)

email_body = """
I hope you are doing well. I am writing to inform you about the current status of our customer analytics dashboard project. We have completed approximately 75% of the development work, including the user authentication module, dashboard interface, and initial data visualization features. However, we are experiencing delays in integrating the real-time data processing pipeline due to unexpected API compatibility issues.

The development team estimates that resolving these issues and completing the remaining integration work will require an additional four working days. Therefore, I would like to request an extension of the project submission deadline from October 14 to October 20, 2026. This additional time will also allow us to conduct comprehensive testing, fix any bugs, and ensure that the final product meets the agreed quality standards.

Meanwhile, the design team has completed the revised user interface, and the documentation team is preparing the technical documentation and user guide. We will continue working on these tasks in parallel to minimize the impact of the delay.

Please let me know whether the proposed extension is acceptable. I can also arrange a brief meeting tomorrow to discuss the revised timeline, outstanding issues, and resource requirements.

Thank you for your understanding and support.
"""
print(summarize_one("PASTE SUBJECT HERE", "sender@company.com", email_body))

[input tokens: 249, min_new_tokens: 40]
The email thread discusses the status of the customer analytics dashboard project. The development team estimates that resolving these issues and completing the remaining integration work will require an additional four working days. The project submission deadline will be from October 14 to October 20, 2026. The design team has completed the revised user interface, and the documentation team is preparing the technical documentation and user guide. The team will continue working on these tasks to minimize the impact of the delay.
